# Volume 8, Appendix F: Quadratic forms and quadrics (Python companion)

Exercises first, solutions at the end. We compute signatures in two independent ways, classify random quadric surfaces, find the quadric through three skew lines, and count points on quadrics over finite fields, where the discriminant decides between $(p+1)^2$ and $p^2+1$. We use `numpy`, `sympy`, `itertools` and `matplotlib`.

*Before each solution, predict what the output will look like.*

In [1]:
import itertools, math, random
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
rng = np.random.default_rng(6); random.seed(6)

## Exercises

**P6.1 (★) Two ways to a signature.** For random symmetric integer matrices, compute the signature by counting eigenvalue signs, and independently by the diagonalisation of Proposition 6.1 done in exact arithmetic (symmetric Gaussian elimination). Check that they always agree (Theorem 6.2) and test Sylvester's criterion (Exercise 6.3).

**P6.2 (★) Which quadric?** Write a function that takes $(M,b,c)$ and returns the type of the quadric $x^{\mathsf T}Mx+2b^{\mathsf T}x+c=0$ following the proof of Theorem 6.4. Classify $x^2+2xy+3y^2-z^2+2z=0$, $xy+yz+zx=1$, $x^2+y^2=2z+1$ and $xy=z$.

**P6.3 (★★) Three skew lines.** Choose three random pairwise skew lines in $\mathbb R^3$, find the quadric through them (Exercise 6.12), check that its $4\times4$ matrix has rank $4$ and signature $(2,2)$, and find a few lines meeting all three; check that they lie on the quadric.

**P6.4 (★★) Counting over $\mathbb F_p$.** For random nondegenerate quadratic forms in four variables over $\mathbb F_p$ ($p=3,5,7$), count the points of the quadric surface in $\mathbb P^3(\mathbb F_p)$. Check that the count is $(p+1)^2$ or $p^2+1$ (Theorem 6.11) and find a rule, in terms of the determinant, that predicts which one. Check also that nondegenerate conics have $p+1$ points (Exercise 6.10).

## Solutions

### P6.1
**How to find it.** Symmetric elimination: if $a_{11}\ne0$, subtract multiples of the first row *and* column to clear them; if the diagonal is zero but $a_{ij}\ne0$, first replace $e_i$ by $e_i+e_j$ (then $q(e_i+e_j)=2a_{ij}\ne0$). This is the proof of Proposition 6.1 made into an algorithm.

In [2]:
def diagonalise(A):
    A = sp.Matrix(A); n = A.shape[0]; diag = []
    while n:
        if all(A[i, j] == 0 for i in range(n) for j in range(n)):
            diag += [0] * n; break
        k = next((i for i in range(n) if A[i, i] != 0), None)
        if k is None:                                   # zero diagonal: e_i -> e_i + e_j
            i, j = next((i, j) for i in range(n) for j in range(n) if A[i, j] != 0)
            P = sp.eye(n); P[j, i] = 1; A = P.T * A * P; k = i
        P = sp.eye(n); P.col_swap(0, k); A = P.T * A * P     # move it to position 0
        a = A[0, 0]; diag.append(a)
        E = sp.eye(n)
        for j in range(1, n): E[0, j] = -A[0, j] / a
        A = (E.T * A * E)[1:, 1:]; n -= 1
    return diag
def sig_diag(A):
    d = diagonalise(A); return (sum(1 for x in d if x > 0), sum(1 for x in d if x < 0))
def sig_eig(A):
    ev = np.linalg.eigvalsh(np.array(A, float)); return (int((ev > 1e-9).sum()), int((ev < -1e-9).sum()))
agree = 0
for _ in range(300):
    n = random.randint(2, 5)
    B = rng.integers(-3, 4, (n, n)); A = (B + B.T).tolist()
    agree += sig_diag(A) == sig_eig(A)
print("agreement:", agree, "of 300")
ok = 0
for _ in range(300):
    n = random.randint(2, 5); B = rng.integers(-3, 4, (n, n)); A = sp.Matrix(B + B.T) + random.randint(0, 6) * sp.eye(n)
    crit = all(A[:k, :k].det() > 0 for k in range(1, n + 1))
    ok += crit == (sig_diag(A) == (n, 0))
print("Sylvester's criterion correct:", ok, "of 300")
print("xy+yz+zx:", sig_diag([[0, sp.Rational(1, 2), sp.Rational(1, 2)], [sp.Rational(1, 2), 0, sp.Rational(1, 2)], [sp.Rational(1, 2), sp.Rational(1, 2), 0]]))

agreement: 300 of 300
Sylvester's criterion correct: 300 of 300
xy+yz+zx: (1, 2)


The two computations agree every time: the number of positive squares in *any* diagonalisation equals the number of positive eigenvalues, as Sylvester's law predicts. The criterion is right in all cases, and $xy+yz+zx$ has signature $(1,2)$ (Exercise 6.1).

### P6.2
**How to find it.** Follow the two cases of the proof. With $M$ invertible the sign of $c'=b^{\mathsf T}M^{-1}b-c$ and the signature of $M$ decide; with $\operatorname{rank}M=2$, the signature of $M$ decides between the two paraboloids.

In [3]:
def quadric_type(M, b, c):
    M = sp.Matrix(M); b = sp.Matrix(b)
    Ah = M.row_join(b).col_join(b.T.row_join(sp.Matrix([[c]])))
    if Ah.det() == 0: return "degenerate"
    if M.det() != 0:
        cp = (b.T * M.inv() * b)[0] - c
        p, m = sig_diag(M / cp)
        return {3: "ellipsoid", 2: "hyperboloid of one sheet", 1: "hyperboloid of two sheets", 0: "empty"}[p]
    p, m = sig_diag(M)
    return "elliptic paraboloid" if p * m == 0 else "hyperbolic paraboloid"
h = sp.Rational(1, 2)
print(quadric_type([[1, 1, 0], [1, 3, 0], [0, 0, -1]], [0, 0, 1], 0))
print(quadric_type([[0, h, h], [h, 0, h], [h, h, 0]], [0, 0, 0], -1))
print(quadric_type([[1, 0, 0], [0, 1, 0], [0, 0, 0]], [0, 0, -1], -1))
print(quadric_type([[0, h, 0], [h, 0, 0], [0, 0, 0]], [0, 0, -h], 0))

hyperboloid of two sheets
hyperboloid of two sheets
elliptic paraboloid
hyperbolic paraboloid


The first is $(x+y)^2+2y^2-(z-1)^2=-1$, a hyperboloid of two sheets. The second, $xy+yz+zx=1$, has $M$ of signature $(1,2)$ and $c'=1$: two sheets as well. The third is the elliptic paraboloid $z=\frac12(x^2+y^2-1)$ and the fourth the saddle.

### P6.3
**How to find it.** Sample three points on each line, solve the $9\times10$ linear system for the coefficients, and build the symmetric $4\times4$ matrix. A line meeting $L_1,L_2,L_3$: through a point $X$ of $L_1$, the lines meeting $L_2$ form the plane $\langle X,L_2\rangle$; it meets $L_3$ in one point $Y$, and the line $XY$ meets all three.

In [4]:
def rand_line():
    return rng.normal(size=3), rng.normal(size=3)
def skew(L, M):
    (p, u), (q, v) = L, M
    return abs(np.linalg.det(np.array([q - p, u, v]))) > 1e-3
while True:
    Ls = [rand_line() for _ in range(3)]
    if all(skew(a, b) for a, b in itertools.combinations(Ls, 2)): break
mons = lambda x, y, z: [x * x, y * y, z * z, x * y, y * z, z * x, x, y, z, 1.0]
rows = [mons(*(p + t * u)) for p, u in Ls for t in (-1.0, 0.3, 2.0)]
_, s, Vt = np.linalg.svd(np.array(rows)); co = Vt[-1]
a, b, c, d, e, f, g, hh, i, j = co
Q = np.array([[a, d / 2, f / 2, g / 2], [d / 2, b, e / 2, hh / 2], [f / 2, e / 2, c, i / 2], [g / 2, hh / 2, i / 2, j]])
ev = np.linalg.eigvalsh(Q)
print("the 9 singular values, all nonzero (rank 9, so a one-dimensional null space):", np.round(s, 4))
print("rank 4, signature:", (int((ev > 1e-9).sum()), int((ev < -1e-9).sum())))
val = lambda X: np.array(mons(*X)) @ co
print("max |q| on the three lines:", max(abs(val(p + t * u)) for p, u in Ls for t in np.linspace(-5, 5, 21)))
(p1, u1), (p2, u2), (p3, u3) = Ls
for t in (-1.0, 0.0, 1.5):
    X = p1 + t * u1
    nrm = np.cross(p2 - X, u2)                       # normal of the plane through X and L2
    s3 = (nrm @ (X - p3)) / (nrm @ u3); Y = p3 + s3 * u3
    w = Y - X
    meets2 = abs(np.linalg.det(np.array([p2 - X, u2, w]))) < 1e-9
    print(f"transversal from t={t}: meets L2 {meets2}, max |q| on it {max(abs(val(X + r * w)) for r in np.linspace(-3, 3, 13)):.2e}")

the 9 singular values, all nonzero (rank 9, so a one-dimensional null space): [4.61459e+01 9.46120e+00 7.96970e+00 3.77700e+00 3.07090e+00 1.91740e+00
 7.94200e-01 1.43300e-01 2.77000e-02]
rank 4, signature: (2, 2)
max |q| on the three lines: 3.441691376337985e-14
transversal from t=-1.0: meets L2 True, max |q| on it 1.67e-14
transversal from t=0.0: meets L2 True, max |q| on it 7.72e-15
transversal from t=1.5: meets L2 True, max |q| on it 1.08e-14


The $9\times10$ system has rank $9$, so a one-dimensional solution space: exactly one quadric, the quadric has signature $(2,2)$, the ruled type, and the transversals lie on it: they are the second family of lines.

### P6.4
**How to find it.** Brute force over $\mathbb F_p^4$ is only $p^4$ vectors. For the rule, recall that the hyperbolic quadric is $x_0x_3-x_1x_2$, of determinant $\frac1{16}$, a square. Test whether $\det A$ is a square modulo $p$.

In [5]:
def count_proj(A, p, n):
    cnt = sum(1 for v in itertools.product(range(p), repeat=n) if any(v) and int(np.array(v) @ A @ np.array(v)) % p == 0)
    return cnt // (p - 1)
def is_square(a, p): return pow(a % p, (p - 1) // 2, p) == 1
for p in (3, 5, 7):
    seen = {}
    for _ in range(40 if p < 7 else 15):
        while True:
            B = np.array([[random.randrange(p) for _ in range(4)] for _ in range(4)]); A = (B + B.T) % p
            dt = int(round(np.linalg.det(A))) % p
            if dt: break
        N = count_proj(A, p, 4)
        seen.setdefault(("det square" if is_square(dt, p) else "det non-square", N), 0)
        seen[("det square" if is_square(dt, p) else "det non-square", N)] += 1
    print(p, "(p+1)^2 =", (p + 1) ** 2, " p^2+1 =", p * p + 1, " observed:", seen)
for p in (3, 5, 7, 11):
    counts = set()
    for _ in range(30):
        while True:
            B = np.array([[random.randrange(p) for _ in range(3)] for _ in range(3)]); A = (B + B.T) % p
            if int(round(np.linalg.det(A))) % p: break
        counts.add(count_proj(A, p, 3))
    print("conics over F_%d: point counts %s" % (p, counts))

3 (p+1)^2 = 16  p^2+1 = 10  observed: {('det non-square', 10): 16, ('det square', 16): 24}
5 (p+1)^2 = 36  p^2+1 = 26  observed: {('det non-square', 26): 27, ('det square', 36): 13}


7 (p+1)^2 = 64  p^2+1 = 50  observed: {('det non-square', 50): 6, ('det square', 64): 9}
conics over F_3: point counts {4}
conics over F_5: point counts {6}
conics over F_7: point counts {8}
conics over F_11: point counts {12}


Every count is $(p+1)^2$ or $p^2+1$, and the determinant decides: a square determinant gives the hyperbolic quadric, a non-square the elliptic one. (In general the hyperbolic case is "discriminant $(-1)^{n/2}\det$ is a square"; in dimension $4$ the sign is $+1$.) Every nondegenerate conic has exactly $p+1$ points.